<div style="display: flex; align-items: center; gap: 16px;">
  
  <h1 style="color: #4A2D6F; font-weight: 800; margin: 0; flex: 1; text-align: center;">Microsoft Foundry: Agent Framework + OTEL — PoC</h1>
</div>
<p align="center" style="color: #666; font-style: italic; margin-top: -4px;">End-to-end proof of concept — from agent execution to trace validation in Log Analytics</p>

Build and interact with an AI agent using Microsoft Agent Framework, Microsoft Foundry, and OpenTelemetry.
The notebook is intentionally split into two execution paths:

1. **Storytelling + Microsoft Learn via MCP** — routed through the latest Microsoft Agent Framework SDKs.
2. **Microsoft Sentinel Data Exploration via MCP** — preserved on the existing Foundry project-connected MCP dependency by design.

For observability, the notebook wires up **Azure Monitor / OpenTelemetry tracing** to send agentic telemetry to:

- Microsoft Foundry — Tracing (Preview)
- Application Insights
- Interrogate Log Analytics (AppDependencies)

---

<h2 style="color: #D83B01;">Prerequisites</h2>

<details>
<summary><strong>Expand Prerequisites</strong></summary>

Before running this notebook, ensure the following are in place:

**1. Azure CLI Installed**

The notebook uses `DefaultAzureCredential`, which relies on Azure CLI for local authentication.

- **Install via winget (Windows):**
  ```powershell
  winget install --id Microsoft.AzureCLI -e --accept-source-agreements --accept-package-agreements
  ```
- **Other platforms / manual install:** [https://aka.ms/installazurecli](https://aka.ms/installazurecli)

**2. Logged in via Azure CLI with Appropriate Permissions**

After installing, sign in and verify your session:
```bash
az login
az account show
```
Your Entra ID identity must have **Contributor** (or equivalent) access to the Microsoft Foundry project and the associated Application Insights resource.

**3. Microsoft Foundry Project with Application Insights**

You need a project provisioned in **Microsoft Foundry** that is connected to an **Application Insights** instance backed by a **Log Analytics workspace**. The notebook retrieves the Application Insights connection string from the project at runtime to enable telemetry export.

**4. Microsoft Sentinel MCP Project Connection**

The Microsoft Sentinel section intentionally depends on an existing Foundry **project connection** for the Sentinel MCP endpoint. Keep that connection in place if you want the Sentinel cells to run.

</details>

---


<h2 style="color: #0078D4;">0. Create or Reuse Virtual Environment &amp; Register Kernel</h2>

<ul>
  <li>Create (or reuse) a Python virtual environment to isolate dependencies for this project, then install <code>ipykernel</code> and register it as a notebook kernel.</li>
  <li>Once setup is complete, select <strong>Kernel</strong> and choose your new <code>.venv</code> environment.</li>
</ul>


In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), ".venv")

# Create the virtual environment (safe to run repeatedly)
subprocess.check_call([sys.executable, "-m", "venv", venv_dir])

# Resolve the venv Python path per OS, then use "python -m pip" for reliability
venv_python = (
    os.path.join(venv_dir, "Scripts", "python.exe")
    if os.name == "nt"
    else os.path.join(venv_dir, "bin", "python")
)

# Install / upgrade ipykernel inside the venv
subprocess.check_call([venv_python, "-m", "pip", "install", "--upgrade", "ipykernel"])

# Register the venv as a Jupyter kernel
subprocess.check_call([
    venv_python,
    "-m",
    "ipykernel",
    "install",
    "--user",
    "--name",
    "ai-agent-demo",
    "--display-name",
    "AI Agent Demo (.venv)",
])

print(f"Virtual environment created or reused at {venv_dir}")
print("Select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker, then continue.")

<h3 style="color: #D83B01;">Activate or Deactivate the Python Virtual Environment</h3>

Run the cell below to **activate** the <code style="color: #2EA043;">.venv</code> so it is recognized as an available kernel.

You only need to create the environment once (Section 0), but you may need to re-activate it after restarting VS Code or opening a new terminal session.

To **deactivate**, run <code style="color: #D29922;">deactivate</code> in your PowerShell terminal.


In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), ".venv")
venv_dir_norm = os.path.normcase(os.path.normpath(venv_dir))

active_venv = os.environ.get("VIRTUAL_ENV", "")
active_venv_norm = os.path.normcase(os.path.normpath(active_venv)) if active_venv else ""

already_active = (
    active_venv_norm == venv_dir_norm
    or os.path.normcase(os.path.normpath(sys.prefix)) == venv_dir_norm
)

if already_active:
    print(f"ℹ️ .venv is already active: {venv_dir}")
else:
    # Resolve the activation script path per OS
    if os.name == "nt":
        activate_script = os.path.join(venv_dir, "Scripts", "Activate.ps1")
        subprocess.check_call(["powershell", "-ExecutionPolicy", "Bypass", "-File", activate_script])
    else:
        activate_script = os.path.join(venv_dir, "bin", "activate")
        subprocess.check_call(["bash", "-c", f"source {activate_script}"])

    print(f"✅ Virtual environment activated: {activate_script}")

print("You can now select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker.")

<h2 style="color: #0078D4;">1. Install Python Packages &amp; Dependencies</h2>

Install the required Python packages.

<details>
<summary><strong>Package Overview (Expand to view dependencies)</strong></summary>
<br>

🤖 <code style="color: #0078D4; font-weight: 600;">agent-framework-core</code>
 — Core Microsoft Agent Framework package for defining and running agents.<br>
💬 <code style="color: #0078D4; font-weight: 600;">agent-framework-openai</code>
 — Current Agent Framework OpenAI Responses client package. In this notebook it is configured for Azure OpenAI using the deployed endpoint and model metadata.<br>
📦 <code style="color: #0078D4; font-weight: 600;">azure-ai-projects</code>
 — Still used for Foundry project metadata, the Application Insights connection string, and the preserved Sentinel MCP dependency.<br>
🔐 <code style="color: #0078D4; font-weight: 600;">azure-identity</code>
 — Provides <code>DefaultAzureCredential</code> for seamless Azure authentication via Entra ID.<br>
📡 <code style="color: #0E7C6B; font-weight: 600;">opentelemetry-sdk</code>
 — Core OpenTelemetry SDK for instrumentation and tracing.<br>
📊 <code style="color: #0078D4; font-weight: 600;">azure-monitor-opentelemetry</code>
 — Azure Monitor exporter and tracing setup for notebook telemetry.<br>
&emsp;↳ <code style="color: #888; font-weight: 500;">azure-core-tracing-opentelemetry</code>
 <span style="color: #666; font-size: 0.9em;">Transitive dep — Azure SDK tracing bridge.</span>

</details>

In [ ]:
import json
import subprocess
import sys

outdated = subprocess.check_output(
    [sys.executable, "-m", "pip", "list", "--outdated", "--format=json"],
    text=True,
)
if any(pkg["name"].lower() == "pip" for pkg in json.loads(outdated)):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

%pip --disable-pip-version-check install --upgrade --pre azure-monitor-opentelemetry-exporter
%pip --disable-pip-version-check install --upgrade "agent-framework-core>=1.0.0" "agent-framework-openai>=1.0.0" "azure-ai-projects>=2.0.0b4"
%pip --disable-pip-version-check install --upgrade azure-identity azure-monitor-opentelemetry azure-core-tracing-opentelemetry opentelemetry-sdk

<h3 style="color: #D83B01;">Confirm Existing Azure / Microsoft Foundry Deployment</h3>

Load the latest <code style="color: #2EA043;">build_info-&lt;suffix&gt;.json</code> and print the current infrastructure summary before importing the Azure/Foundry libraries.

<table style="margin-top: 8px; border: none; border-collapse: collapse;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📂</td>
    <td style="border: none; padding: 4px 0; color: #555;">Reads deployment metadata (resource group, Key Vault, App Insights, model, endpoints)</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">✅</td>
    <td style="border: none; padding: 4px 0; color: #555;">Confirms all resources are provisioned before proceeding</td>
  </tr>
</table>


In [ ]:
import json
from pathlib import Path

def resolve_build_info_path() -> Path:
    candidates = sorted(
        Path(".").glob("build_info-*.json"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        r"No build_info-<suffix>.json file found. Run deployment/deploy-foundry-env.ps1 first to generate one."
    )

build_info_path = resolve_build_info_path()
build_info = json.loads(build_info_path.read_text(encoding="utf-8"))
foundry_proj_ep = build_info["foundry_project_endpoint"]
genai_model = build_info["genai_model"]
signed_in_account = globals().get("signed_in_account")

def show_current_build_status(
    build_info: dict,
    *,
    signed_in_account: str | None = None,
    app_insights_connection_status: str = "This project only ✅",
    law_rbac_status: str = "Log Analytics Reader on DIBSecCom ✅",
) -> None:
    user_status = (
        f"{signed_in_account} added to zolab-ai-dev ✅"
        if signed_in_account
        else "Signed-in account not available"
    )
    rows = [
        ("☁️ Resource Group", f"✅ {build_info['rg']}"),
        ("🗄️ Storage", build_info["storage_account"]),
        ("🔐 Key Vault", build_info["key_vault"]),
        ("📊 App Insights", build_info["appinsights"]),
        ("🤖 AI Foundry", build_info["foundry_name"]),
        ("🏢 AI Project", build_info["foundry_project_name"]),
        ("🧠 Model", f"{build_info['genai_model']} (GlobalStandard)"),
        ("📝 Build Info", build_info_path.name),
        ("🔗 App Insights Connection", app_insights_connection_status),
        ("📡 LAW RBAC", law_rbac_status),
        ("👤 User", user_status),
        ("🔌 Foundry Project Endpoint", build_info["foundry_project_endpoint"]),
        ("🤖 Azure OpenAI Endpoint", build_info["azure_openai_endpoint"]),
    ]

    item_width = max(29, max(len(item) for item, _ in rows) + 2)
    status_width = max(78, max(len(status) for _, status in rows) + 2)

    print()
    print("● ☁️🎉🚀 Your infrastructure is all green!")
    print()
    print("┌" + ("─" * item_width) + "┬" + ("─" * status_width) + "┐")
    print("│" + " Item".ljust(item_width) + "│" + " Status".ljust(status_width) + "│")
    print("├" + ("─" * item_width) + "┼" + ("─" * status_width) + "┤")
    for item, status in rows:
        print("│" + f" {item}".ljust(item_width) + "│" + f" {status}".ljust(status_width) + "│")
    print("└" + ("─" * item_width) + "┴" + ("─" * status_width) + "┘")
    print()
    print("Ready for the notebook! 🎯")

show_current_build_status(build_info, signed_in_account=signed_in_account)


<h2 style="color: #0078D4;">2. Import Libraries</h2>

Import the necessary classes from the Azure, Foundry, Agent Framework, and OpenTelemetry SDKs.

<details>
<summary><strong>What Each Import Does</strong></summary>
<br>

🔐 <code style="color: #0078D4; font-weight: 600;">DefaultAzureCredential</code>
 <span style="color: #888; font-size: 0.88em;">azure-identity</span>
 — Picks up credentials from your environment (Azure CLI, managed identity, etc.).<br>
🤖 <code style="color: #0078D4; font-weight: 600;">AIProjectClient</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Client for Foundry project metadata, telemetry connection details, and the preserved Sentinel MCP dependency.<br>
💬 <code style="color: #0078D4; font-weight: 600;">OpenAIChatClient</code>
 <span style="color: #888; font-size: 0.88em;">agent-framework-openai</span>
 — Agent Framework Responses client used for the main storytelling and Microsoft Learn workflow, configured against Azure OpenAI.<br>
✨ <code style="color: #0078D4; font-weight: 600;">PromptAgentDefinition</code>
 <span style="color: #888; font-size: 0.88em;">azure-ai-projects</span>
 — Still used for the Sentinel-specific Foundry project agent that depends on the existing MCP project connection.<br>
📡 <code style="color: #0E7C6B; font-weight: 600;">enable_instrumentation</code>
 <span style="color: #888; font-size: 0.88em;">agent-framework-core</span>
 — Enables Agent Framework observability hooks for traces and model/tool events.<br>
📊 <code style="color: #0E7C6B; font-weight: 600;">configure_azure_monitor</code>
 <span style="color: #888; font-size: 0.88em;">azure-monitor-opentelemetry</span>
 — Sets up Azure Monitor as the OpenTelemetry export backend.

</details>

<details>
<summary><strong>How the Pieces Wire Together</strong></summary>
<br>

The notebook now connects Foundry, Agent Framework, and the telemetry stack in four steps:<br>

<span style="color: #0078D4; font-weight: 700;">①</span>
 <code style="color: #0078D4;">project_client.telemetry.get_application_insights_connection_string()</code>
 — retrieves the telemetry destination from the Foundry project.<br>
<span style="color: #0078D4; font-weight: 700;">②</span>
 <code style="color: #0078D4;">configure_azure_monitor(...)</code>
 — initializes the Azure Monitor OpenTelemetry pipeline.<br>
<span style="color: #0078D4; font-weight: 700;">③</span>
 <code style="color: #0E7C6B;">enable_instrumentation(...)</code>
 — enables Agent Framework telemetry code paths for the main notebook agent.<br>
<span style="color: #0078D4; font-weight: 700;">④</span>
 Manual spans (<code style="color: #D83B01;">create_agent</code>, <code style="color: #D83B01;">invoke_agent</code>)
 — add notebook-specific business context for troubleshooting and correlation.<br>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🏗️</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Foundry SDK</strong> — project setup, telemetry connection string retrieval, and the preserved Sentinel MCP connection path.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🤖</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Agent Framework</strong> — main notebook agent runtime for storytelling and Microsoft Learn grounded responses over Azure OpenAI.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Telemetry Stack</strong> — <code>opentelemetry-sdk</code> (tracing runtime) → <code>azure-core-tracing-opentelemetry</code> (bridge) → <code>azure-monitor-opentelemetry</code> (exporter).</td>
  </tr>
</table>

</details>

<details>
<summary><strong>MSFT Learn References</strong></summary>

- [Microsoft Agent Framework](https://github.com/microsoft/agent-framework)
- [Agent Framework observability samples](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/observability)
- [Azure Monitor](https://learn.microsoft.com/python/api/overview/azure/monitor-opentelemetry-readme?view=azure-python#getting-started)

</details>


In [ ]:
import sys, platform

from agent_framework.openai import OpenAIChatClient
from agent_framework.observability import create_resource, enable_instrumentation
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential

print("✅ All libraries imported successfully")
print(f"   💻 Platform: {platform.system()} {platform.release()}")
print(f"   🐍 Python:   {sys.version.split()[0]}")

<h2 style="color: #0078D4;">3. Configure Credentials, Project Client, and Agent Framework Client</h2>

- Reuse the deployment values loaded in the **Confirm Existing Deployment** section above.
- Create an authenticated `AIProjectClient` for Foundry metadata and the retained Sentinel MCP dependency.
- Create an authenticated Azure OpenAI-backed `OpenAIChatClient` for the main Agent Framework workflow.
- Confirm which credential was selected for this notebook session.


In [ ]:
import json
import logging
import os
import shutil
import subprocess
from pathlib import Path

# build_info and build_info_path are set in Section 0
foundry_proj_ep = build_info["foundry_project_endpoint"]
azure_openai_endpoint = build_info["azure_openai_endpoint"]
genai_model = build_info["genai_model"]
signed_in_account = globals().get("signed_in_account")

# ---------------------------------------------------------------------------
def _install_az_cli() -> bool:
    """Attempt to install Azure CLI using winget. Returns True on success."""
    if os.name != "nt":
        print("❌ Automatic install via winget is only supported on Windows.")
        print("   👉 Install Azure CLI from: https://aka.ms/installazurecli")
        return False

    if not shutil.which("winget"):
        print("❌ winget is not available on this system.")
        print("   👉 Install Azure CLI manually from: https://aka.ms/installazurecli")
        return False

    print("📦 Installing Azure CLI via winget … (this may take a few minutes)")
    result = subprocess.run(
        ["winget", "install", "--id", "Microsoft.AzureCLI", "-e", "--accept-source-agreements", "--accept-package-agreements"],
        capture_output=True, text=True, timeout=300,
    )
    if result.returncode == 0:
        print("✅ Azure CLI installed successfully.")
        az_default = os.path.join(os.environ.get("ProgramFiles", r"C:\Program Files"), "Microsoft SDKs", "Azure", "CLI2", "wbin")
        if os.path.isdir(az_default) and az_default not in os.environ.get("PATH", ""):
            os.environ["PATH"] = az_default + os.pathsep + os.environ["PATH"]
            print(f"   Added {az_default} to PATH for this session.")
        return True

    print(f"❌ winget install failed (exit code {result.returncode}).")
    if result.stderr:
        print(f"   {result.stderr.strip()}")
    print("   👉 Try installing manually from: https://aka.ms/installazurecli")
    return False


# ---------------------------------------------------------------------------
# Helper: check for Azure CLI and attempt interactive login if needed
# ---------------------------------------------------------------------------
def _ensure_az_login() -> bool:
    """Return True if Azure CLI is available and the user is logged in."""
    az_exe = "az.cmd" if os.name == "nt" else "az"

    if not shutil.which(az_exe):
        print("⚠️  Azure CLI is NOT installed.")
        if not _install_az_cli():
            return False
        if not shutil.which(az_exe):
            print("❌ Azure CLI still not found on PATH after install.")
            print("   Please restart VS Code so the updated PATH takes effect, then re-run this cell.")
            return False

    print("🔍 Azure CLI found on PATH — checking login status …")

    check = subprocess.run(
        [az_exe, "account", "show"],
        capture_output=True, text=True, timeout=15,
    )
    if check.returncode == 0:
        print("✅ Already logged in to Azure CLI.")
        return True

    print("⚠️  No active Azure CLI session detected. Launching 'az login' …")
    print("   A browser window should open — please sign in with your Azure account.\n")
    login = subprocess.run([az_exe, "login"], capture_output=True, text=True, timeout=120)
    if login.returncode == 0:
        if login.stdout:
            print(login.stdout.strip())
        print("\n✅ Azure CLI login succeeded.")
        return True

    print(f"\n❌ 'az login' did not complete successfully (exit code {login.returncode}).")
    if login.stderr:
        print(f"   {login.stderr.strip()}")
    print("   Please run 'az login' manually in a terminal, then re-run this cell.")
    return False


# ---------------------------------------------------------------------------
# Authenticate
# ---------------------------------------------------------------------------
credential = DefaultAzureCredential()

_identity_handler = logging.StreamHandler()
identity_logger = logging.getLogger("azure.identity")
identity_logger.addHandler(_identity_handler)
identity_logger.setLevel(logging.INFO)

signed_in_account = None

try:
    _ = credential.get_token("https://management.azure.com/.default")

except Exception as auth_ex:
    print(f"⚠️  DefaultAzureCredential failed: {type(auth_ex).__name__}")
    print("   Falling back to Azure CLI authentication …\n")

    if not _ensure_az_login():
        raise SystemExit(
            "🛑 Cannot proceed without valid Azure credentials. "
            "Please install/login to Azure CLI and re-run this cell."
        )

    credential = DefaultAzureCredential()
    _ = credential.get_token("https://management.azure.com/.default")

finally:
    identity_logger.removeHandler(_identity_handler)
    identity_logger.setLevel(logging.WARNING)

project_client = AIProjectClient(
    endpoint=foundry_proj_ep,
    credential=credential,
)

azure_openai_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "preview")
main_chat_client = OpenAIChatClient(
    model=genai_model,
    azure_endpoint=azure_openai_endpoint,
    credential=credential,
    api_version=azure_openai_api_version,
)

os.environ["AZURE_AI_PROJECT_ENDPOINT"] = foundry_proj_ep
os.environ["AZURE_OPENAI_ENDPOINT"] = azure_openai_endpoint
os.environ["AZURE_OPENAI_MODEL"] = genai_model
os.environ["AZURE_OPENAI_CHAT_MODEL"] = genai_model
os.environ["AZURE_OPENAI_API_VERSION"] = azure_openai_api_version

_lw = 18
print("\n✅ Notebook clients configured")
print(f"   🌐 {'Project endpoint:':<{_lw}} {foundry_proj_ep}")
print(f"   ☁️ {'Azure OpenAI:':<{_lw}} {azure_openai_endpoint}")
print(f"   🧠 {'Model deployment:':<{_lw}} {genai_model}")
print(f"   📄 {'Build info:':<{_lw}} {build_info_path.resolve()}")
print(f"   🔐 {'Auth:':<{_lw}} DefaultAzureCredential")
print(f"   🤖 {'Main runtime:':<{_lw}} Agent Framework (OpenAIChatClient on Azure OpenAI)")
print(f"   🛰️ {'Sentinel path:':<{_lw}} Foundry project agent + MCP project connection")


# ---------------------------------------------------------------------------
# Optional: surface which credential + identity is in use
# ---------------------------------------------------------------------------
def _run_command(command: list[str]) -> str | None:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if result.returncode == 0:
            value = (result.stdout or "").strip()
            return value or None
    except Exception:
        pass
    return None


def _resolve_identity_hint(credential_name: str) -> str | None:
    if credential_name == "AzureCliCredential":
        az_exe = "az.cmd" if os.name == "nt" else "az"
        return _run_command([az_exe, "account", "show", "--query", "user.name", "-o", "tsv"])
    if credential_name == "AzurePowerShellCredential":
        powershell_cmd = "$ctx = Get-AzContext; if ($ctx -and $ctx.Account) { $ctx.Account.Id }"
        return _run_command(["pwsh", "-NoProfile", "-Command", powershell_cmd])
    return None


try:
    selected_credential = getattr(credential, "_successful_credential", None)
    if selected_credential is not None:
        credential_name = selected_credential.__class__.__name__
        credential_name_color = globals().get("ansi_magenta", "\033[95m")
        credential_ansi_reset = globals().get("ansi_reset", "\033[0m")
        credential_name_colored = f"{credential_name_color}{credential_name}{credential_ansi_reset}"
        print(f"      🔐 Credential used: {credential_name_colored}")

        identity_hint = _resolve_identity_hint(credential_name)
        if identity_hint:
            signed_in_account = identity_hint
            print(f"      👤 Signed-in account: {identity_hint}")
        else:
            print("      👤 Signed-in account: not available for this credential type")
    else:
        print("      Credential Type: check azure.identity INFO logs above")
except Exception as ex:
    print(f"      Credential probe skipped: {type(ex).__name__}: {ex}")

<h2 style="color: #0078D4;">3.1 Enable AI Agent (client-side) Observability/Telemetry</h2>

<details>
<summary><strong>Tracing Setup Overview</strong></summary>

Configure tracing for both the Agent Framework runtime and the retained Foundry/Sentinel path:
1. **`configure_azure_monitor`** — sets up the OpenTelemetry pipeline that exports notebook traces to Application Insights
2. **`enable_instrumentation`** — enables Agent Framework observability hooks for model and tool operations
3. **`project_client.telemetry.get_application_insights_connection_string()`** — keeps the telemetry destination grounded in the Foundry project metadata
4. **Manual notebook spans** — add run-specific context (`create_agent`, `invoke_agent`, `persist_story`, `persist_sentinel_query`) for correlation

**Note:**
- The notebook keeps `AIProjectClient` for project metadata and the Sentinel MCP dependency.
- The main storytelling and Microsoft Learn path uses Agent Framework instrumentation.
- The Sentinel section remains a separate project-agent execution path by design.

</details>

In [ ]:
import os
from html import escape
from uuid import uuid4

from azure.core.settings import settings
from IPython.display import HTML, display
from opentelemetry import baggage, context as otel_context, trace

settings.tracing_implementation = "opentelemetry"

project_name = foundry_proj_ep.rstrip("/").split("/")[-1] if "foundry_proj_ep" in globals() else "unknown-project"
os.environ["OTEL_SERVICE_NAME"] = "foundry-agent-framework-demo"
os.environ["OTEL_SERVICE_VERSION"] = "2026.04.05"

if "telemetry_session_id" not in globals():
    telemetry_session_id = str(uuid4())

# Historical notebook override: keep Azure VM/App Service resource detectors out of local runs
# so the SDK does not probe IMDS when the notebook is not running on Azure.
azure_host_markers = (
    "WEBSITE_SITE_NAME",
    "WEBSITE_HOSTNAME",
    "FUNCTIONS_WORKER_RUNTIME",
    "KUBERNETES_SERVICE_HOST",
    "AKS_ARM_NAMESPACE_ID",
)
running_on_azure_host = any(os.environ.get(name) for name in azure_host_markers)

if not running_on_azure_host:
    os.environ["OTEL_EXPERIMENTAL_RESOURCE_DETECTORS"] = "otel"
    os.environ["OTEL_RESOURCE_DETECTORS"] = "otel"
    os.environ["APPLICATIONINSIGHTS_STATSBEAT_DISABLED_ALL"] = "true"

os.environ["OTEL_RESOURCE_ATTRIBUTES"] = (
    f"service.namespace=foundry-agent-demo,service.instance.id={telemetry_session_id},"
    f"deployment.environment=demo,foundry.project.name={project_name}"
)

# It is important to set the above environment variables before configuring Azure Monitor.
from azure.monitor.opentelemetry import configure_azure_monitor

application_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

if not globals().get("_agent_framework_otel_initialized", False):
    configure_azure_monitor(
        connection_string=application_insights_connection_string,
        resource=create_resource(),
        sampling_ratio=1.0,
    )
    enable_instrumentation(enable_sensitive_data=True)
    globals()["_agent_framework_otel_initialized"] = True
else:
    print("ℹ️ Agent Framework telemetry was already initialized in this kernel session.")

tracer = trace.get_tracer("foundry_agent_framework_notebook")


def make_baggage_context(values: dict[str, object]):
    context = otel_context.get_current()
    for key, value in values.items():
        if value is None:
            continue
        value_text = str(value).strip()
        if value_text:
            context = baggage.set_baggage(key, value_text, context=context)
    return context


# -----------------------------------------------------------------------------
# Notebook status output
# -----------------------------------------------------------------------------
accent_style = "color: #C239B3; font-weight: 700;"
secondary_style = "color: #0078D4; font-weight: 700;"
local_mode_text = (
    "Local notebook mode - Azure resource detectors disabled; Statsbeat disabled"
    if not running_on_azure_host
    else "Azure-host environment detected - default Azure resource detection enabled"
)

display(
    HTML(
        f"""
<div style="font-family: Consolas, 'Cascadia Code', monospace; line-height: 1.5;">
  <div>Tracing enabled -&gt; Application Insights for project: '<span style=\"{accent_style}\">{escape(project_name)}</span>'</div>
  <div>- [connection string retrieved from: <span style=\"{accent_style}\">Microsoft Foundry</span>]</div>
  <div>- OTEL service name: <span style=\"{secondary_style}\">{escape(os.environ['OTEL_SERVICE_NAME'])}</span></div>
  <div>- Telemetry session id: {escape(str(telemetry_session_id))}</div>
  <div>- Agent Framework instrumentation: Enabled</div>
  <div>- Content recording: Enabled</div>
  <div>- Trace sampling: 100%</div>
  <div>- Runtime mode: {escape(local_mode_text)}</div>
</div>
"""
    )
)

<h3 style="color: #0078D4;">3.2 Configure MSFT Learn MCP Tool</h3>

Run this step to create the public remote MCP tool descriptor that the main Agent Framework agent will use in Step 4.

<details>
<summary><strong>MSFT Learn MCP Tool Setup</strong></summary>

- MSFT Learn MCP documentation: [Azure MCP Server docs](https://learn.microsoft.com/azure/developer/azure-mcp-server/)
- MSFT Learn MCP URL: [Microsoft Learn MCP Endpoint](https://learn.microsoft.com/api/mcp)
- This step builds the remote MCP tool payload by using the installed `OpenAIChatClient.get_mcp_tool(...)` helper from Agent Framework.
- The public Learn endpoint does not need a Foundry project connection, so the notebook can attach it directly to the main agent.
- The Microsoft Sentinel MCP dependency remains separate in Step 3.3 because that path requires an existing Foundry project connection for OAuth passthrough.

</details>

In [ ]:
# ---------------------------------------------------------------------------
# MSFT Learn MCP Tool Spec (used by the Step 4 Agent Framework agent)
# ---------------------------------------------------------------------------
msft_learn_mcp_url = "https://learn.microsoft.com/api/mcp"
print(f"MSFT Learn MCP URL: {msft_learn_mcp_url}")

mcp_tool_spec = OpenAIChatClient.get_mcp_tool(
    name="msft-learn",
    url=msft_learn_mcp_url,
    approval_mode="never_require",
)

print("Configured Agent Framework MCP tool: msft-learn")

<h3 style="color: #0078D4;">3.3 Configure Microsoft Sentinel Data Exploration MCP Tool</h3>

Run this step if you want the agent to use Microsoft Sentinel data exploration in addition to Microsoft Learn.

<details>
<summary><strong>Why this section is different</strong></summary>

- Microsoft Learn works as a public MCP endpoint, so the notebook can declare it directly.
- Microsoft Sentinel Data Exploration uses OAuth identity passthrough, so Foundry needs a project connection for the MCP endpoint.
- This cell uses `az rest` to look for an existing Foundry project connection that already points at the Sentinel MCP endpoint.
- If the connection exists, the notebook builds `sentinel_mcp_tool_spec` for Step 4.
- If it does not exist yet, the cell prints the exact project resource path and the expected connection name/ID so you can provision it and rerun the cell.

</details>

In [ ]:
# ---------------------------------------------------------------------------
# Microsoft Sentinel Data Exploration MCP Tool Spec (optional, OAuth passthrough)
# ---------------------------------------------------------------------------
import json
import os
import subprocess

from azure.ai.projects.models import MCPTool

az_cmd = "az.cmd" if os.name == "nt" else "az"
sentinel_mcp_url = "https://sentinel.microsoft.com/mcp/data-exploration"
sentinel_connection_name = os.environ.get("SENTINEL_MCP_CONNECTION_NAME", "MicrosoftSentinelData").strip()
sentinel_project_connection_id = os.environ.get("SENTINEL_MCP_PROJECT_CONNECTION_ID", "").strip()
sentinel_mcp_tool_spec = None

def _run_az_json(command: list[str]) -> dict:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        stderr = (result.stderr or "").strip()
        raise RuntimeError(stderr or f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    stdout = (result.stdout or "").strip()
    return json.loads(stdout) if stdout else {}

def _run_az_tsv(command: list[str]) -> str:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        stderr = (result.stderr or "").strip()
        raise RuntimeError(stderr or f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    return (result.stdout or "").strip()

foundry_account_id = _run_az_tsv(
    [
        az_cmd,
        "resource",
        "show",
        "--resource-group",
        build_info["rg"],
        "--name",
        build_info["foundry_name"],
        "--resource-type",
        "Microsoft.CognitiveServices/accounts",
        "--query",
        "id",
        "-o",
        "tsv",
    ]
)
project_resource_id = f"{foundry_account_id}/projects/{build_info['foundry_project_name']}"
connections_uri = f"https://management.azure.com{project_resource_id}/connections?api-version=2025-06-01"

print(f"Sentinel MCP URL: {sentinel_mcp_url}")
print(f"Foundry project resource: {project_resource_id}")

try:
    connections_response = _run_az_json([az_cmd, "rest", "--method", "get", "--uri", connections_uri])
    connections = connections_response.get("value", [])
except Exception as ex:
    connections = []
    print(f"⚠️ Unable to enumerate project connections via Azure CLI: {type(ex).__name__}: {ex}")

matching_connection = None
if sentinel_project_connection_id:
    matching_connection = next(
        (connection for connection in connections if connection.get("id", "").lower() == sentinel_project_connection_id.lower()),
        None,
    )

if matching_connection is None:
    matching_connection = next(
        (
            connection
            for connection in connections
            if connection.get("name", "").lower() == sentinel_connection_name.lower()
            or connection.get("properties", {}).get("target", "").rstrip("/").lower() == sentinel_mcp_url.rstrip("/").lower()
        ),
        None,
    )

if matching_connection is not None:
    sentinel_project_connection_id = matching_connection["id"]

if sentinel_project_connection_id:
    sentinel_mcp_tool_spec = MCPTool(
        server_label="microsoft-sentinel-data",
        server_url=sentinel_mcp_url,
        require_approval="always",
        project_connection_id=sentinel_project_connection_id,
    )
    print("✅ Microsoft Sentinel Data Exploration MCP is enabled for this notebook.")
    print(f"   Project connection ID: {sentinel_project_connection_id}")
else:
    print("⚠️ No Foundry project connection was found for the Sentinel MCP endpoint.")
    print(f"   Expected connection name: {sentinel_connection_name}")
    print(f"   CLI check: az rest --method get --uri \"{connections_uri}\"")
    print("   Set SENTINEL_MCP_PROJECT_CONNECTION_ID to an existing connection ID and rerun this cell.")
    print("   This notebook can attach the MCP tool once the Foundry project connection already exists.")

<h2 style="color: #0078D4;">4. Create the AI Agents</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">✨</td>
    <td style="border: none; padding: 4px 0; color: #555;">Main agent uses Microsoft Agent Framework with the Azure OpenAI Responses client for storytelling and Microsoft Learn retrieval</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔧</td>
    <td style="border: none; padding: 4px 0; color: #555;">MSFT Learn MCP is attached directly to the main Agent Framework agent as a public remote MCP tool</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🛡️</td>
    <td style="border: none; padding: 4px 0; color: #555;">If a Foundry project connection exists, the notebook also creates a separate Sentinel project agent so the Sentinel MCP dependency stays on its existing project-connected path</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔄</td>
    <td style="border: none; padding: 4px 0; color: #555;">The Sentinel helper still uses <code style="color: #0078D4;">create_version</code> because that flow depends on the Foundry project connection and OAuth passthrough</td>
  </tr>
</table>


In [ ]:
from uuid import uuid4
from opentelemetry.trace import SpanKind, Status, StatusCode

if "main_chat_client" not in globals():
    raise RuntimeError("Run Section 3 first so the Azure OpenAI-backed OpenAIChatClient is configured.")

main_agent_name = "ZoDEfendersAgent-1702"
model_name = genai_model

main_agent_instructions = (
    "You are an illuminating cybernetic storytelling agent. "
    "You craft engaging 6-sentence stories based on user prompts and context. "
    "Be sure to use vivid language and creative scenarios to captivate the reader. "
    "Use the full names of any characters you create in the story. "
    "You are also a practical assistant for learning Microsoft technologies. "
    "When the user asks factual Microsoft questions, use the tools available to you and ground the answer in Microsoft Learn."
 )

sentinel_agent_instructions = (
    "You are a Microsoft Sentinel investigation assistant. "
    "Use the Microsoft Sentinel Data Exploration MCP tool when it is available through the connected Foundry project resource. "
    "Return concise, investigation-ready findings and clearly call out when more validation is required."
 )

# Shared run correlation id across notebook steps (created once per kernel session)
demo_run_id = globals().get("demo_run_id") or str(uuid4())
globals()["demo_run_id"] = demo_run_id

main_tool_labels = [mcp_tool_spec["server_label"]]
main_agent_creation_context = make_baggage_context(
    {
        "demo-run-id": demo_run_id,
        "agent-name": main_agent_name,
        "model-name": model_name,
        "project-name": globals().get("project_name", "unknown-project"),
        "telemetry-session-id": globals().get("telemetry_session_id", ""),
        "agent-runtime": "agent-framework",
    }
)

with tracer.start_as_current_span(
    f"create_agent {main_agent_name}",
    kind=SpanKind.CLIENT,
    context=main_agent_creation_context,
) as span:
    span.set_attribute("gen_ai.operation.name", "create_agent")
    span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
    span.set_attribute("gen_ai.request.model", model_name)
    span.set_attribute("gen_ai.agent.name", main_agent_name)
    span.set_attribute("gen_ai.agent.description", main_agent_instructions)
    span.set_attribute("demo.run_id", demo_run_id)
    span.set_attribute("app.session.id", globals().get("telemetry_session_id", ""))
    span.set_attribute("agent.runtime", "microsoft.agent_framework")
    span.set_attribute("agent.tools.count", len(main_tool_labels))
    span.set_attribute("agent.tools.labels", ",".join(main_tool_labels))
    span.add_event("create_agent.start")

    try:
        main_agent = main_chat_client.as_agent(
            name=main_agent_name,
            instructions=main_agent_instructions,
            tools=[mcp_tool_spec],
        )
        span.add_event("create_agent.success")
    except Exception as ex:
        span.record_exception(ex)
        span.set_attribute("error.type", type(ex).__name__)
        span.set_status(Status(StatusCode.ERROR, str(ex)))
        span.add_event("create_agent.failure")
        raise

globals()["main_agent"] = main_agent
print(f"Main Agent Framework agent configured: {main_agent_name}")
print(f"Attached tools: {', '.join(main_tool_labels)}")
print(f"Run correlation id: {demo_run_id}")

sentinel_project_agent = None
sentinel_agent_reference_payload = None
sentinel_tool = globals().get("sentinel_mcp_tool_spec")

if sentinel_tool is not None:
    sentinel_agent_name = f"{main_agent_name}-sentinel"
    sentinel_creation_context = make_baggage_context(
        {
            "demo-run-id": demo_run_id,
            "agent-name": sentinel_agent_name,
            "model-name": model_name,
            "project-name": globals().get("project_name", "unknown-project"),
            "telemetry-session-id": globals().get("telemetry_session_id", ""),
            "agent-runtime": "azure-ai-project-agent",
            "mcp-server": "microsoft-sentinel-data",
        }
    )

    with tracer.start_as_current_span(
        f"create_agent {sentinel_agent_name}",
        kind=SpanKind.CLIENT,
        context=sentinel_creation_context,
    ) as span:
        span.set_attribute("gen_ai.operation.name", "create_agent")
        span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
        span.set_attribute("gen_ai.request.model", model_name)
        span.set_attribute("gen_ai.agent.name", sentinel_agent_name)
        span.set_attribute("gen_ai.agent.description", sentinel_agent_instructions)
        span.set_attribute("demo.run_id", demo_run_id)
        span.set_attribute("app.session.id", globals().get("telemetry-session-id", globals().get("telemetry_session_id", "")))
        span.set_attribute("agent.runtime", "azure.ai.projects")
        span.set_attribute("agent.tools.count", 1)
        span.set_attribute("agent.tools.labels", "microsoft-sentinel-data")
        span.add_event("create_agent.start")

        try:
            sentinel_project_agent = project_client.agents.create_version(
                agent_name=sentinel_agent_name,
                definition=PromptAgentDefinition(
                    model=model_name,
                    instructions=sentinel_agent_instructions,
                    tools=[sentinel_tool],
                ),
            )
            span.set_attribute("gen_ai.agent.id", sentinel_project_agent.id)
            span.set_attribute("gen_ai.agent.version", str(sentinel_project_agent.version))
            span.add_event("create_agent.success")
        except Exception as ex:
            span.record_exception(ex)
            span.set_attribute("error.type", type(ex).__name__)
            span.set_status(Status(StatusCode.ERROR, str(ex)))
            span.add_event("create_agent.failure")
            raise

    sentinel_agent_reference_payload = {
        "agent_reference": {
            "name": sentinel_project_agent.name,
            "id": sentinel_project_agent.id,
            "type": "agent_reference",
        }
    }

    globals()["sentinel_project_agent"] = sentinel_project_agent
    globals()["sentinel_agent_reference_payload"] = sentinel_agent_reference_payload
    print(
        f"Sentinel project agent configured: {sentinel_project_agent.name} "
        f"(id: {sentinel_project_agent.id}, version: {sentinel_project_agent.version})"
    )
else:
    globals()["sentinel_project_agent"] = None
    globals()["sentinel_agent_reference_payload"] = None
    print("Sentinel project agent not created because no Foundry project connection is configured.")

<h2 style="color: #0078D4;">5. Query the Main Agent Framework Agent</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📖</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Pass 1</strong> — generate a fictional story from the main Agent Framework storytelling persona</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔍</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Pass 2</strong> — retrieve factual guidance from Microsoft Learn through the public MCP tool attached to the main Agent Framework agent</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🧭</td>
    <td style="border: none; padding: 4px 0; color: #555;">The Microsoft Sentinel MCP interaction stays in the next cell because it intentionally uses the separate Foundry project-connected Sentinel agent</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">💾</td>
    <td style="border: none; padding: 4px 0; color: #555;">This cell appends the story and Microsoft Learn results to <code style="color: #2EA043;">stories.json</code> with Agent Framework run metadata</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;">Each interaction is wrapped in <code style="color: #0E7C6B;">OpenTelemetry</code> spans for trace correlation</td>
  </tr>
</table>


In [ ]:
import json
import re
from datetime import datetime, timezone
from html import escape
from pathlib import Path
from uuid import uuid4

from IPython.display import HTML, display
from opentelemetry.trace import SpanKind, Status, StatusCode

if "main_agent" not in globals():
    raise RuntimeError("Run Section 4 first so the main Agent Framework agent is configured.")

story_prompt = (
    "Write about a Cloud Solutions Architect - Security by the name of 'Azure Zo' in Frankfurt, Germany. "
    "He is a man who moonlights as a cybernetic superhero called 'Die DEfender', protector of the digital realm. "
    "Include a plot twist in the last sentence. Always refer to the two characters by their full names."
 )

facts_prompt = (
    "Use the MSFT Learn MCP tool to explain what Microsoft Foundry is for a Cloud Solutions Architect - Security. "
    "Return concise factual guidance only with this exact structure:\n"
    "MSFT Learn Insights:\n"
    "- 3 to 5 key points\n"
    "- Security and governance considerations\n"
    "- Practical next steps\n"
    "Sources:\n"
    "- Include at least one Microsoft Learn URL"
 )

main_agent_display_name = getattr(main_agent, "name", globals().get("main_agent_name", "ZoDEfendersAgent-1702"))
main_tool_labels = globals().get("main_tool_labels", ["msft-learn"])

def extract_agent_framework_text(result) -> str:
    if isinstance(result, str):
        return result.strip()

    text = getattr(result, "text", None)
    if isinstance(text, str) and text.strip():
        return text.strip()

    parts = []
    for message in getattr(result, "messages", None) or []:
        for content in getattr(message, "content", None) or []:
            candidate = getattr(content, "text", None)
            if isinstance(candidate, str) and candidate.strip():
                parts.append(candidate.strip())
                continue

            candidate_value = getattr(candidate, "value", None)
            if isinstance(candidate_value, str) and candidate_value.strip():
                parts.append(candidate_value.strip())
                continue

            fallback_value = getattr(content, "value", None)
            if isinstance(fallback_value, str) and fallback_value.strip():
                parts.append(fallback_value.strip())

    if parts:
        return "\n".join(parts).strip()

    return str(result).strip()

async def run_main_agent_interaction(*, interaction_name: str, prompt: str):
    interaction_id = str(uuid4())
    interaction_context = make_baggage_context(
        {
            "demo-run-id": globals().get("demo_run_id", ""),
            "agent-name": main_agent_display_name,
            "interaction-name": interaction_name,
            "interaction-id": interaction_id,
            "model-name": model_name,
            "project-name": globals().get("project_name", "unknown-project"),
            "telemetry-session-id": globals().get("telemetry_session_id", ""),
            "agent-runtime": "agent-framework",
        }
    )

    with tracer.start_as_current_span(
        f"invoke_agent {main_agent_display_name}",
        kind=SpanKind.CLIENT,
        context=interaction_context,
    ) as interaction_span:
        interaction_span.set_attribute("demo.run_id", globals().get("demo_run_id", ""))
        interaction_span.set_attribute("gen_ai.operation.name", "invoke_agent")
        interaction_span.set_attribute("gen_ai.provider.name", "azure.ai.openai")
        interaction_span.set_attribute("gen_ai.request.model", model_name)
        interaction_span.set_attribute("gen_ai.agent.name", main_agent_display_name)
        interaction_span.set_attribute("app.interaction", interaction_name)
        interaction_span.set_attribute("app.interaction.id", interaction_id)
        interaction_span.set_attribute("app.prompt", prompt)
        interaction_span.set_attribute("app.session.id", globals().get("telemetry_session_id", ""))
        interaction_span.set_attribute("agent.runtime", "microsoft.agent_framework")
        interaction_span.set_attribute("agent.tools.labels", ",".join(main_tool_labels))
        interaction_span.add_event(
            "interaction.start",
            {
                "app.interaction": interaction_name,
                "app.interaction.id": interaction_id,
            },
        )

        try:
            result = await main_agent.run(prompt)
            response_text = extract_agent_framework_text(result)

            response_id = getattr(result, "id", None)
            if response_id:
                interaction_span.set_attribute("gen_ai.response.id", response_id)

            response_model = getattr(result, "model", None)
            if response_model:
                interaction_span.set_attribute("gen_ai.response.model", response_model)

            interaction_span.set_attribute("app.completion", response_text[:500] if response_text else "")
            interaction_span.add_event(
                "response.completed",
                {
                    "app.interaction": interaction_name,
                    "app.interaction.id": interaction_id,
                },
            )
            return result, response_text, interaction_id
        except Exception as ex:
            interaction_span.record_exception(ex)
            interaction_span.set_attribute("error.type", type(ex).__name__)
            interaction_span.set_status(Status(StatusCode.ERROR, str(ex)))
            interaction_span.add_event("response.failure")
            raise

async def run_main_agent_demo():
    run_context = make_baggage_context(
        {
            "demo-run-id": globals().get("demo_run_id", ""),
            "agent-name": main_agent_display_name,
            "model-name": model_name,
            "project-name": globals().get("project_name", "unknown-project"),
            "telemetry-session-id": globals().get("telemetry_session_id", ""),
            "agent-runtime": "agent-framework",
        }
    )

    with tracer.start_as_current_span("agent-query", context=run_context) as run_span:
        run_span.set_attribute("demo.run_id", globals().get("demo_run_id", ""))
        run_span.set_attribute("agent.name", main_agent_display_name)
        run_span.set_attribute("gen_ai.request.model", model_name)
        run_span.set_attribute("app.session.id", globals().get("telemetry_session_id", ""))
        run_span.set_attribute("agent.runtime", "microsoft.agent_framework")

        print("\n=== Pass 1: Generate fictional story (Agent Framework) ===")
        story_result, story_text, story_interaction_id = await run_main_agent_interaction(
            interaction_name="story",
            prompt=story_prompt,
        )

        print("\n=== Pass 2: Retrieve factual Microsoft Learn insights (Agent Framework + MCP) ===")
        facts_result, facts_text, facts_interaction_id = await run_main_agent_interaction(
            interaction_name="facts",
            prompt=facts_prompt,
        )

        output_sections = []
        if story_text:
            output_sections.append("Story:\n" + story_text.strip())
        if facts_text:
            output_sections.append(facts_text.strip())

        assistant_text = "\n\n".join(section for section in output_sections if section).strip()
        interaction_ids = {
            "story": story_interaction_id,
            "facts": facts_interaction_id,
        }

        run_span.set_attribute("app.output.sections", len(output_sections))
        run_span.set_attribute("app.interaction.count", len(interaction_ids))
        run_span.set_attribute("app.completion", assistant_text[:500] if assistant_text else "")

        return story_result, story_text, facts_result, facts_text, assistant_text, interaction_ids

(
    story_result,
    story_text,
    facts_result,
    facts_text,
    assistant_text,
    interaction_ids,
 ) = await run_main_agent_demo()

def append_story(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            stories = json.load(file)
        if not isinstance(stories, list):
            stories = []
    except (FileNotFoundError, json.JSONDecodeError):
        stories = []

    next_id = max((item.get("id", 0) for item in stories), default=0) + 1
    payload["id"] = next_id
    stories.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(stories, file, indent=2, ensure_ascii=False)

    return next_id

stories_file = Path("stories.json")
persist_context = make_baggage_context(
    {
        "demo-run-id": globals().get("demo_run_id", ""),
        "agent-name": main_agent_display_name,
        "model-name": model_name,
        "project-name": globals().get("project_name", "unknown-project"),
        "telemetry-session-id": globals().get("telemetry_session_id", ""),
        "agent-runtime": "agent-framework",
    }
)

with tracer.start_as_current_span("persist_story", context=persist_context) as persist_span:
    story_id = append_story(
        stories_file,
        {
            "entry_type": "agent_framework_main",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "agent": main_agent_display_name,
            "agent_runtime": "agent-framework",
            "model": model_name,
            "prompt": story_prompt,
            "story": story_text,
            "msft_learn_insights": facts_text,
            "combined_output": assistant_text,
            "tool_labels": main_tool_labels,
            "interaction_ids": interaction_ids,
            "foundry_project_endpoint": build_info["foundry_project_endpoint"],
            "resource_group": build_info["rg"],
            "requested_by": build_info.get("requested_by"),
            "demo_run_id": globals().get("demo_run_id", ""),
        },
    )
    persist_span.set_attribute("app.story.id", story_id)
    persist_span.set_attribute("app.stories.path", str(stories_file))
    persist_span.set_attribute("app.interaction.count", len(interaction_ids))

def render_query_html(response_text: str, interaction_map: dict[str, str]) -> None:
    formatted_text = escape((response_text or "").strip())
    formatted_text = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", formatted_text)
    formatted_text = formatted_text.replace("\n", "<br>")

    interaction_items = "".join(
        f"<li><strong>{escape(name.title())}</strong>: <code>{escape(interaction_id)}</code></li>"
        for name, interaction_id in interaction_map.items()
    )
    interaction_html = (
        f"<ul style=\"margin: 0; padding-left: 18px;\">{interaction_items}</ul>"
        if interaction_items
        else "Not available"
    )

    display(
        HTML(
            f"""
<div style="margin-top: 16px; border: 1px solid #d0d7de; border-radius: 14px; overflow: hidden; font-family: 'Segoe UI', Arial, sans-serif;">
  <div style="background: linear-gradient(135deg, #2b7fff, #6b4ce6); color: white; padding: 14px 18px;">
    <div style="font-size: 18px; font-weight: 700;">Fictional Story + Microsoft Learn Result</div>
    <div style="font-size: 12px; opacity: 0.9; margin-top: 4px;">Agent Framework main path with independent story and documentation interactions</div>
  </div>
  <div style="padding: 18px; background: #ffffff; color: #1f2328;">
    <div style="line-height: 1.7; font-size: 14px;">{formatted_text}</div>
    <hr style="border: 0; border-top: 1px solid #d8dee4; margin: 18px 0;">
    <div style="display: grid; grid-template-columns: 180px 1fr; row-gap: 10px; column-gap: 12px; font-size: 13px;">
      <div style="font-weight: 600; color: #57606a;">Interaction ids</div>
      <div>{interaction_html}</div>
      <div style="font-weight: 600; color: #57606a;">Record saved</div>
      <div><code>{story_id}</code> in <code>{escape(str(stories_file))}</code></div>
      <div style="font-weight: 600; color: #57606a;">MCP server</div>
      <div><code>msft-learn</code></div>
    </div>
  </div>
</div>
"""
        )
    )

if not assistant_text:
    print("⚠️ No assistant text was returned.")
else:
    render_query_html(assistant_text, interaction_ids)

<h3 style="color: #0078D4;">5.1 Query Microsoft Sentinel Data Exploration MCP</h3>

This call remains separate from the main Agent Framework story and Microsoft Learn path because the Sentinel flow intentionally stays on the Foundry project-connected MCP dependency.

<details>
<summary><strong>What this cell does</strong></summary>

- Uses the Sentinel-specific project agent created in Step 4 when a Foundry project connection is available.
- Keeps the Microsoft Sentinel MCP dependency on the existing project connection and OAuth passthrough path by design.
- Wraps the interaction in dedicated OpenTelemetry spans so the Sentinel trace is easy to isolate in Foundry and Log Analytics.
- Persists the result to <code style="color: #2EA043;">stories.json</code> as a separate notebook record.
- Handles MCP approvals and OAuth consent inside this cell so it does not depend on the main Agent Framework query helpers.

</details>

In [ ]:
import json
import re
from datetime import datetime, timezone
from html import escape
from pathlib import Path

from IPython.display import HTML, display

if globals().get("sentinel_project_agent") is None:
    raise RuntimeError(
        "Microsoft Sentinel Data Exploration MCP is not configured. Run Sections 3.3 and 4, then rerun this cell."
    )

sentinel_project_agent = globals()["sentinel_project_agent"]
sentinel_agent_reference_payload = globals().get("sentinel_agent_reference_payload") or {
    "agent_reference": {
        "name": sentinel_project_agent.name,
        "id": sentinel_project_agent.id,
        "type": "agent_reference",
    }
}

sentinel_prompt = (
    "When and where was the last time Lorenzo logged in? "
    "Use the SignInLogs table and/or related tables to query. "
    "Ensure the Time, City, State, Country, IP, and what application or resource, for example 'Azure Portal', is provided."
 )

MAX_APPROVAL_ROUNDS = 5
conversation_ids = globals().setdefault("conversation_ids", {})

def parse_response(response) -> tuple[str, list[str], list[str], object, list[str]]:
    output_items = getattr(response, "output", None) or []
    output_types = [getattr(item, "type", type(item).__name__) for item in output_items]

    approval_ids = [
        item.id
        for item in output_items
        if getattr(item, "type", None) == "mcp_approval_request" and getattr(item, "id", None)
    ]

    consent_links = [
        item.consent_link
        for item in output_items
        if getattr(item, "type", None) == "oauth_consent_request" and getattr(item, "consent_link", None)
    ]

    text_parts = []
    for item in output_items:
        for content in getattr(item, "content", None) or []:
            text_obj = getattr(content, "text", None)
            if isinstance(text_obj, str) and text_obj.strip():
                text_parts.append(text_obj)
            elif text_obj is not None:
                text_value = getattr(text_obj, "value", None)
                if isinstance(text_value, str) and text_value.strip():
                    text_parts.append(text_value)

    text = (getattr(response, "output_text", None) or "\n".join(text_parts)).strip()
    return text, approval_ids, output_types, getattr(response, "incomplete_details", None), consent_links

def run_query_with_auto_approval(openai_client, agent_reference_payload, prompt, interaction_name, interaction_span, max_rounds=5):
    conversation = openai_client.conversations.create()
    print(f"Conversation created: {conversation.id}")
    interaction_span.add_event(
        "conversation.created",
        {
            "app.conversation.id": conversation.id,
            "app.interaction": interaction_name,
        },
    )

    baggage_values = {
        "demo-run-id": globals().get("demo_run_id", ""),
        "agent-name": sentinel_project_agent.name,
        "agent-id": sentinel_project_agent.id,
        "agent-version": getattr(sentinel_project_agent, "version", ""),
        "interaction-name": interaction_name,
        "conversation-id": conversation.id,
        "model-name": model_name,
        "telemetry-session-id": globals().get("telemetry_session_id", ""),
        "mcp-server": "microsoft-sentinel-data",
    }

    def create_agent_response(response_input):
        from opentelemetry import context as otel_context

        baggage_token = otel_context.attach(make_baggage_context(baggage_values))
        try:
            return openai_client.responses.create(
                conversation=conversation.id,
                input=response_input,
                extra_body=agent_reference_payload,
            )
        finally:
            otel_context.detach(baggage_token)

    response = create_agent_response(prompt)
    approvals_granted = 0

    for round_number in range(1, max_rounds + 1):
        text, approval_ids, output_types, incomplete_details, consent_links = parse_response(response)
        status = getattr(response, "status", "unknown")

        print(f"Response status: {status}")
        print(f"Response output types: {output_types}")
        if incomplete_details:
            print(f"Incomplete details: {incomplete_details}")
            interaction_span.add_event(
                "response.incomplete",
                {"app.incomplete_details": str(incomplete_details)},
            )

        interaction_span.set_attribute("app.response.status", status)
        interaction_span.set_attribute("app.response.output_types", ",".join(output_types))

        if consent_links:
            unique_consent_links = list(dict.fromkeys(consent_links))
            interaction_span.add_event(
                "oauth.consent.required",
                {"app.oauth.consent.count": len(unique_consent_links)},
            )
            interaction_span.set_attribute("app.oauth.consent_required", True)
            print("OAuth consent is required before the MCP tool can continue:")
            for consent_link in unique_consent_links:
                print(f"- {consent_link}")

            consent_message = "\n".join(
                [
                    "OAuth consent required before the MCP tool can continue.",
                    *unique_consent_links,
                    "Open the consent link, complete sign-in, then rerun this cell.",
                ]
            )
            return conversation.id, response, consent_message, approvals_granted

        if text or not approval_ids:
            interaction_span.add_event(
                "response.completed",
                {
                    "gen_ai.response.id": response.id,
                    "app.approval.rounds": approvals_granted,
                },
            )
            return conversation.id, response, text, approvals_granted

        approvals_granted += len(approval_ids)
        interaction_span.add_event(
            "mcp.approval.auto_approved",
            {
                "app.approval.round": round_number,
                "app.approval.count": len(approval_ids),
            },
        )
        print(f"Approving MCP tool requests (round {round_number}): {approval_ids}")
        response = create_agent_response(
            [
                {
                    "type": "mcp_approval_response",
                    "approve": True,
                    "approval_request_id": request_id,
                }
                for request_id in approval_ids
            ]
        )

    final_text, _, _, _, _ = parse_response(response)
    interaction_span.add_event(
        "response.max_approval_rounds_reached",
        {"app.approval.rounds": approvals_granted},
    )
    print("⚠️ Reached maximum MCP approval rounds without assistant text output.")
    return conversation.id, response, final_text, approvals_granted

with project_client.get_openai_client() as openai_client:
    sentinel_run_context = make_baggage_context(
        {
            "demo-run-id": globals().get("demo_run_id", ""),
            "agent-name": sentinel_project_agent.name,
            "agent-id": sentinel_project_agent.id,
            "agent-version": getattr(sentinel_project_agent, "version", ""),
            "interaction-name": "sentinel",
            "model-name": model_name,
            "project-name": globals().get("project_name", "unknown-project"),
            "telemetry-session-id": globals().get("telemetry_session_id", ""),
            "mcp-server": "microsoft-sentinel-data",
            "agent-runtime": "azure-ai-project-agent",
        }
    )

    with tracer.start_as_current_span("sentinel-agent-query", context=sentinel_run_context) as sentinel_run_span:
        sentinel_run_span.set_attribute("demo.run_id", globals().get("demo_run_id", ""))
        sentinel_run_span.set_attribute("agent.name", sentinel_project_agent.name)
        sentinel_run_span.set_attribute("gen_ai.request.model", model_name)
        sentinel_run_span.set_attribute("app.interaction", "sentinel")
        sentinel_run_span.set_attribute("app.session.id", globals().get("telemetry_session_id", ""))
        sentinel_run_span.set_attribute("mcp.server.label", "microsoft-sentinel-data")
        sentinel_run_span.set_attribute("agent.runtime", "azure.ai.projects")
        sentinel_run_span.set_attribute(
            "mcp.project_connection_id",
            globals().get("sentinel_project_connection_id", ""),
        )

        print("\n=== Pass 3: Retrieve Microsoft Sentinel guidance (independent conversation) ===")
        sentinel_conversation_id, sentinel_response, sentinel_text, approvals_granted = run_query_with_auto_approval(
            openai_client=openai_client,
            agent_reference_payload=sentinel_agent_reference_payload,
            prompt=sentinel_prompt,
            interaction_name="sentinel",
            interaction_span=sentinel_run_span,
            max_rounds=MAX_APPROVAL_ROUNDS,
        )

        conversation_ids["sentinel"] = sentinel_conversation_id
        sentinel_run_span.set_attribute("app.conversation.id", sentinel_conversation_id)
        sentinel_run_span.set_attribute("app.approval.rounds", approvals_granted)
        sentinel_run_span.set_attribute("app.completion", sentinel_text[:500] if sentinel_text else "")
        sentinel_run_span.set_attribute("gen_ai.response.id", sentinel_response.id)
        response_model = getattr(sentinel_response, "model", None)
        if response_model:
            sentinel_run_span.set_attribute("gen_ai.response.model", response_model)

def append_sentinel_record(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            records = json.load(file)
        if not isinstance(records, list):
            records = []
    except (FileNotFoundError, json.JSONDecodeError):
        records = []

    next_id = max((item.get("id", 0) for item in records), default=0) + 1
    payload["id"] = next_id
    records.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(records, file, indent=2, ensure_ascii=False)

    return next_id

stories_file = Path("stories.json")
sentinel_persist_context = make_baggage_context(
    {
        "demo-run-id": globals().get("demo_run_id", ""),
        "agent-name": sentinel_project_agent.name,
        "model-name": model_name,
        "project-name": globals().get("project_name", "unknown-project"),
        "telemetry-session-id": globals().get("telemetry_session_id", ""),
        "interaction-name": "sentinel",
        "mcp-server": "microsoft-sentinel-data",
        "agent-runtime": "azure-ai-project-agent",
    }
)

with tracer.start_as_current_span("persist_sentinel_query", context=sentinel_persist_context) as persist_span:
    sentinel_record_id = append_sentinel_record(
        stories_file,
        {
            "entry_type": "sentinel_mcp",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "agent": sentinel_project_agent.name,
            "agent_runtime": "azure-ai-project-agent",
            "model": model_name,
            "prompt": sentinel_prompt,
            "sentinel_insights": sentinel_text,
            "foundry_project_endpoint": build_info["foundry_project_endpoint"],
            "resource_group": build_info["rg"],
            "requested_by": build_info.get("requested_by"),
            "conversation_id": sentinel_conversation_id,
            "demo_run_id": globals().get("demo_run_id", ""),
            "mcp_server_label": "microsoft-sentinel-data",
            "project_connection_id": globals().get("sentinel_project_connection_id", ""),
        },
    )
    persist_span.set_attribute("app.story.id", sentinel_record_id)
    persist_span.set_attribute("app.stories.path", str(stories_file))
    persist_span.set_attribute("app.interaction", "sentinel")

def render_sentinel_html(response_text: str) -> None:
    formatted_text = escape((response_text or "").strip())
    formatted_text = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", formatted_text)
    formatted_text = formatted_text.replace("\n", "<br>")

    conversation_html = (
        f"<code>{escape(sentinel_conversation_id)}</code>" if sentinel_conversation_id else "Not available"
    )

    display(
        HTML(
            f"""
<div style="margin-top: 16px; border: 1px solid #d0d7de; border-radius: 14px; overflow: hidden; font-family: 'Segoe UI', Arial, sans-serif;">
  <div style="background: linear-gradient(135deg, #0f6cbd, #115ea3); color: white; padding: 14px 18px;">
    <div style="font-size: 18px; font-weight: 700;">Microsoft Sentinel Result</div>
    <div style="font-size: 12px; opacity: 0.9; margin-top: 4px;">Foundry project-connected Sentinel MCP path preserved by design</div>
  </div>
  <div style="padding: 18px; background: #ffffff; color: #1f2328;">
    <div style="line-height: 1.7; font-size: 14px;">{formatted_text}</div>
    <hr style="border: 0; border-top: 1px solid #d8dee4; margin: 18px 0;">
    <div style="display: grid; grid-template-columns: 180px 1fr; row-gap: 10px; column-gap: 12px; font-size: 13px;">
      <div style="font-weight: 600; color: #57606a;">Conversation</div>
      <div>{conversation_html}</div>
      <div style="font-weight: 600; color: #57606a;">Record saved</div>
      <div><code>{sentinel_record_id}</code> in <code>{escape(str(stories_file))}</code></div>
      <div style="font-weight: 600; color: #57606a;">MCP server</div>
      <div><code>microsoft-sentinel-data</code></div>
    </div>
  </div>
</div>
"""
        )
    )

render_sentinel_html(sentinel_text)

<h2 style="color: #0078D4;">6. Validate Observability (Traces) in Log Analytics</h2>

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">✅</td>
    <td style="border: none; padding: 4px 0; color: #555;">Confirm agentic telemetry is populating in Log Analytics</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔗</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>End-to-end view</strong> — joined dependencies and traces by <code style="color: #0078D4;">OperationId</code></td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📊</td>
    <td style="border: none; padding: 4px 0; color: #555;"><strong>Runs-only trend</strong> — call volume, failures, and latency percentiles in 15-min bins</td>
  </tr>
</table>

<details>
<summary><strong>Environment &amp; Query Prerequisites</strong></summary>
<br>

🗂️ <strong>Workspace resource ID</strong><br>
<code style="color: #888; font-size: 0.88em;">/subscriptions/&lt;subscription-id&gt;/resourceGroups/Sentinel/providers/Microsoft.OperationalInsights/workspaces/&lt;workspace-name&gt;</code><br><br>

⚠️ The Azure CLI command <code style="color: #D83B01;">az monitor log-analytics query</code> may fail in some environments due to extension/runtime mismatch.<br>
🔐 Fallback uses direct HTTPS calls to <code style="color: #0078D4;">api.loganalytics.io</code> with a <code style="color: #0078D4;">DefaultAzureCredential</code> bearer token.

</details>


In [ ]:
import json
import subprocess
import os
import urllib.error
import urllib.request

from azure.identity import DefaultAzureCredential

# Dynamically resolve the Security subscription ID via Azure CLI
az_cmd = "az.cmd" if os.name == "nt" else "az"
workspace_subscription_id = subprocess.check_output(
    [az_cmd, "account", "list", "--query", "[?name=='Security'].id | [0]", "-o", "tsv"],
    text=True,
).strip()
workspace_resource_group = "Sentinel"
workspace_name = "DIBSecCom"
workspace_resource_id = (
    f"/subscriptions/{workspace_subscription_id}/resourceGroups/{workspace_resource_group}"
    f"/providers/Microsoft.OperationalInsights/workspaces/{workspace_name}"
)

# Dynamically resolve the workspace customer ID (required by Log Analytics Query API)
workspace_customer_id = subprocess.check_output(
    [az_cmd, "monitor", "log-analytics", "workspace", "show",
     "--resource-group", workspace_resource_group,
     "--workspace-name", workspace_name,
     "--subscription", workspace_subscription_id,
     "--query", "customerId", "-o", "tsv"],
    text=True,
).strip()
print(f"Using Log Analytics workspace: {workspace_resource_id}")
print(f"Workspace customer ID: {workspace_customer_id}")


# Optional: filter to one run when demo_run_id exists in the current kernel session
demo_run_filter = globals().get("demo_run_id")
if isinstance(demo_run_filter, str):
    demo_run_filter = demo_run_filter.strip()
else:
    demo_run_filter = ""
run_filter_clause = (
    f'| where tostring(Properties["demo.run_id"]) == "{demo_run_filter}"'
    if demo_run_filter
    else ""
)
if run_filter_clause:
    print(f"Applying run filter for demo.run_id={demo_run_filter}")
else:
    print("No demo_run_id filter applied (showing latest runs in lookback window).")

# End-to-end view including requested fields: agent name, model, system message, host, location
kql_end_to_end = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
__RUN_FILTER__
| where Name in ("agent-query", "responses") or Name startswith "create_agent " or Name startswith "invoke_agent " or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["gen_ai.agent.name"]),
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    system_message = coalesce(
        tostring(Properties["gen_ai.system_instructions"]),
        extract(@'"role"\s*:\s*"system".*?"content"\s*:\s*"([^\"]+)"', 1, tostring(Properties["gen_ai.input.messages"])),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    ),
    demo_run_id = tostring(Properties["demo.run_id"]),
    interaction = tostring(Properties["app.interaction"])
| project
    TimeGenerated,
    Name,
    demo_run_id,
    interaction,
    agent_name,
    model_used,
    system_message,
    computer,
    location,
    Success,
    DurationMs,
    OperationId,
    Prompt = coalesce(tostring(Properties["app.prompt"]), tostring(Properties["gen_ai.prompt"])),
    Completion = coalesce(tostring(Properties["app.completion"]), tostring(Properties["gen_ai.completion"]))
| order by TimeGenerated desc
| take 50
""".replace("__RUN_FILTER__", run_filter_clause).strip()

# Runs-only trend grouped by agent/model/host/location
kql_runs_trend = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
__RUN_FILTER__
| where Name in ("agent-query", "responses") or Name startswith "create_agent " or Name startswith "invoke_agent " or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["gen_ai.agent.name"]),
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    ),
    demo_run_id = tostring(Properties["demo.run_id"]),
    interaction = tostring(Properties["app.interaction"])
| summarize
    Calls = count(),
    Failures = countif(Success == false),
    AvgDurationMs = avg(DurationMs),
    P95DurationMs = percentile(DurationMs, 95)
    by bin(TimeGenerated, 15m), demo_run_id, interaction, agent_name, model_used, computer, location
| order by TimeGenerated desc
""".replace("__RUN_FILTER__", run_filter_clause).strip()

def query_log_analytics(customer_id: str, query: str) -> dict:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://api.loganalytics.io/.default").token

    body = json.dumps({"query": query}).encode("utf-8")
    request = urllib.request.Request(
        url=f"https://api.loganalytics.io/v1/workspaces/{customer_id}/query",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token}",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as ex:
        error_body = ex.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Log Analytics query failed ({ex.code}): {error_body}") from ex

print("Running end-to-end validation query...")
end_to_end_result = query_log_analytics(workspace_customer_id, kql_end_to_end)
print(json.dumps(end_to_end_result, indent=2)[:4000])

print("\nRunning runs-only trend query...")
runs_trend_result = query_log_analytics(workspace_customer_id, kql_runs_trend)
print(json.dumps(runs_trend_result, indent=2)[:4000])